# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string and OpenAI API key in Step 0, then run each cell in order. The notebook creates embeddings for RAG chunks, stores them in Azure DocumentDB, retrieves context with vector and hybrid search, and builds a grounded prompt for a chat model.

The final step prints the prompt your application would send to a chat model.


## Step 0: Connect and configure embeddings

This cell installs dependencies, accepts the DocumentDB connection string and OpenAI API key, and creates both clients.

In [ ]:
import importlib.util, subprocess, sys, os, getpass
for package in ["pymongo", "openai"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
from pymongo import MongoClient
from openai import OpenAI

connection_string = os.environ.get("DOCUMENTDB_CONNECTION_STRING") or getpass.getpass("Paste Azure DocumentDB connection string: ")
openai_api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Paste OpenAI API key: ")
embedding_model = os.environ.get("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

client = MongoClient(connection_string)
db = client["docdbworkshop"]
chunks = db["rag_chunks"]
openai_client = OpenAI(api_key=openai_api_key)
print(db.command({"ping": 1}))
print("Embedding model:", embedding_model)

## Step 1: Create embeddings for RAG chunks

Each chunk is embedded with OpenAI and stored in Azure DocumentDB with its source metadata.

In [ ]:
def create_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(model=embedding_model, input=text)
    return response.data[0].embedding

rag_docs = [
    {"_id":"rag-001","sourceId":"search-module","title":"Vector search","chunk":"Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.","url":"module-4-search","tags":["vector","search"]},
    {"_id":"rag-002","sourceId":"search-module","title":"Full-text search","chunk":"Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.","url":"module-4-search","tags":["full-text","bm25"]},
    {"_id":"rag-003","sourceId":"search-module","title":"Hybrid search","chunk":"Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.","url":"module-4-search","tags":["hybrid","rrf"]},
    {"_id":"rag-004","sourceId":"rag-module","title":"Grounded generation","chunk":"A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.","url":"module-5-rag","tags":["rag","generation"]}
]

chunks.drop()
for doc in rag_docs:
    doc["embedding"] = create_embedding(doc["chunk"])
chunks.insert_many(rag_docs)
embedding_dimensions = len(rag_docs[0]["embedding"])
print("Loaded chunks:", chunks.count_documents({}))
print("Embedding dimensions:", embedding_dimensions)

## Step 2: Create retrieval indexes

The vector index enables semantic retrieval. The BM25 search index helps exact terms like `$search`, `BM25`, and `cosmosSearch` rank correctly.

In [ ]:
db.command({
    "createIndexes": "rag_chunks",
    "indexes": [{
        "name": "idx_chunk_embedding_diskann",
        "key": {"embedding": "cosmosSearch"},
        "cosmosSearchOptions": {"kind": "vector-diskann", "dimensions": embedding_dimensions, "similarity": "COS", "maxDegree": 32, "lBuild": 64}
    }]
})
db.command({
    "createSearchIndexes": "rag_chunks",
    "indexes": [{
        "name": "idx_chunk_fts",
        "definition": {"mappings": {"dynamic": False, "fields": {"chunk": {"type": "string"}}}}
    }]
})

## Step 3: Generate a question embedding and retrieve context

The question is embedded at runtime, then used to retrieve the nearest chunks with `cosmosSearch`.

In [ ]:
question = "How does DocumentDB retrieve context for RAG?"
question_vector = create_embedding(question)
vector_context = list(chunks.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": question_vector, "k": 3}}},
    {"$project": {"_id": 1, "title": 1, "chunk": 1, "url": 1, "score": {"$meta": "searchScore"}}}
]))
vector_context

## Step 4: Hybrid retrieval with RRF

Hybrid retrieval combines BM25 and vector candidates so RAG handles both exact terms and paraphrased questions.

In [ ]:
keyword_context = list(chunks.aggregate([
    {"$search": {"index": "idx_chunk_fts", "text": {"query": question, "path": "chunk"}}},
    {"$limit": 3},
    {"$project": {"_id": 1, "title": 1, "chunk": 1, "url": 1, "score": {"$meta": "searchScore"}}}
]))

def rrf(lists, k=60, top_n=3):
    docs, scores = {}, {}
    for results in lists:
        for rank, doc in enumerate(results):
            doc_id = str(doc["_id"])
            docs[doc_id] = doc
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return [{**docs[doc_id], "rrfScore": score} for doc_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

hybrid_context = rrf([keyword_context, vector_context])
hybrid_context

## Step 5: Build the grounded prompt

This prompt includes retrieved DocumentDB chunks and tells the chat model to answer only from that context.

In [ ]:
context_block = "

".join([f"[{i+1}] {doc['title']}
{doc['chunk']}
Source: {doc['url']}" for i, doc in enumerate(hybrid_context)])
grounded_prompt = f"""You are a helpful assistant for an Azure DocumentDB workshop.
Answer the user's question using only the context below.
If the context does not contain the answer, say you do not know based on the provided context.

<context>
{context_block}
</context>

Question: {question}"""
print(grounded_prompt)